# Triaj — Colab'da QLoRA Fine-tuning

Bu notebook `Qwen2.5-1.5B-Instruct` modelini Triaj'ın kategori/öncelik/duygu/yanıt şemasına QLoRA ile fine-tune eder.

**Önce şunu yap:** Üstteki menüden `Çalışma zamanı (Runtime) → Çalışma zamanı türünü değiştir → T4 GPU` seç. Ücretsiz T4 (16 GB) yeterli.

In [ ]:
!nvidia-smi

## 1) Repoyu çek

`train_qlora.py`, `worker/model.py` içindeki `SYSTEM_PROMPT`'u import ediyor — bu yüzden tüm repoyu klonlamak gerekiyor, sadece `training/` klasörü yetmiyor.

In [ ]:
!git clone --branch claude/proje-gorevleri-b73j2b https://github.com/ZEYDLCN/Triaj.git
%cd Triaj/training

## 2) Bağımlılıklar

Colab'ın kendi torch kurulumuyla çakışmaması için `torch` satırını atlayıp geri kalanı kuruyoruz.

In [ ]:
!grep -v '^torch' requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt

## 3) Veri seti

Şimdilik sentetik şablon veriyle uçtan uca akışı doğruluyoruz. Gerçek kullanımda `prepare_dataset.py`'yi kendi ticket arşivinle değiştir (aynı `text` / `channel` / `customer_tier` / `label` şemasında bir `data/train.jsonl` + `data/val.jsonl` üretmen yeterli).

In [ ]:
!python prepare_dataset.py --n 1500

## 4) Eğitim (QLoRA, 4-bit)

T4'te ~15 dk sürer. `--batch` OOM alırsan 2'ye düşür.

In [ ]:
!python train_qlora.py --epochs 3 --batch 4

## 5) Değerlendirme (base model vs adaptör)

In [ ]:
!python evaluate.py

## 6) Adaptörü indir

Çıktı `output/triaj-qlora` klasöründe. Bunu zipleyip indir, sonra kendi bilgisayarında repodaki `training/output/triaj-qlora` yoluna aç — `docker-compose.yml` worker'a bunu otomatik bağlıyor (`./training/output:/adapters:ro`).

In [ ]:
!zip -r triaj-qlora.zip output/triaj-qlora
from google.colab import files
files.download('triaj-qlora.zip')

### İndirdikten sonra (kendi makinende)

```bash
cd Triaj
unzip ~/Downloads/triaj-qlora.zip -d training/
docker compose restart worker
```

`worker/model.py` adaptör klasörünü görürse otomatik yükler; görmezse base modelle (adaptörsüz) çalışmaya devam eder — sistem her koşulda ayakta kalır, sadece JSON tutarlılığı/kategori isabeti düşer.